# מפגש 22 — פתרון מורה עם בדיקות
מעבדת התלמיד: bill_length_mm/bill_depth_mm, seed17, Test רק בסוף.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
assert len(df)==344
X = df[["bill_length_mm","bill_depth_mm"]].copy()
y = df["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=17,stratify=y)
assert len(X_train)==258 and len(X_test)==86

In [ ]:
candidates=[1,3,5,15,51]
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for k in candidates:
    pipe=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring="f1_macro",return_train_score=True,n_jobs=1)
    assert len(scores["test_score"])==5
    rows.append({"K":k,"train_mean":scores["train_score"].mean(),"cv_mean":scores["test_score"].mean(),"cv_std":scores["test_score"].std(ddof=0)})
results_df=pd.DataFrame(rows)
assert len(results_df)==5
print(results_df)

In [ ]:
best_k=int(results_df.sort_values(["cv_mean","K"],ascending=[False,True]).iloc[0]["K"])
final_model=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),KNeighborsClassifier(n_neighbors=best_k))
final_model.fit(X_train,y_train)
predictions=final_model.predict(X_test)
assert best_k in candidates and len(predictions)==86
print("K",best_k,"macro-F1",f1_score(y_test,predictions,average="macro"),"Accuracy",accuracy_score(y_test,predictions))

In [ ]:
assert all(results_df["cv_std"]>=0)
assert results_df["K"].tolist()==candidates
assert 0<=f1_score(y_test,predictions,average="macro")<=1
assert 0<=accuracy_score(y_test,predictions)<=1
print("Checks passed")

cv_std אינו רווח סמך; test_score של CV הוא אימות פנימי. אין לכוונן את K לאחר ההערכה החיצונית.